# Magnetic ordering

A composition does not determine a structure, and a structure does not determine
a magnetic state. Iron, nickel, cobalt and most of their compounds have several
spin arrangements at nearly the same energy, and which one is lowest changes the
formation energy by tenths of an eV per atom — comfortably enough to move a
material on or off the convex hull.

So a stability screen over magnetic materials has a step before it that a screen
over oxides of aluminium does not: enumerate the orderings, and find out which
one you are actually computing.

This is the shortest tutorial here, and it ends by refusing to answer the
question. That refusal is the point.

In [1]:
import matverse as mv
import numpy as np
import pandas as pd

mv.pl.set_style()

🔬 Starting plot initialization...


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


🧪 Calculators available: 12
    • emt — EMT (LGPL-2.1)
    • lj — Lennard-Jones (LGPL-2.1)
    • mace-mpa — MACE-MPA-0 (MIT)
    • mace-omat — MACE-OMAT-0 (ASL)
    • sevennet — SevenNet (GPL-3.0)
    • chgnet — CHGNet (BSD-3-Clause)
    • m3gnet — M3GNet-PES-MatPES-PBE-2025.2 (BSD-3-Clause)
    • m3gnet-r2scan — M3GNet-PES-MatPES-r2SCAN-2025.2 (BSD-3-Clause)
    • tensornet — TensorNet-PES-MatPES-PBE-2025.2 (BSD-3-Clause)
    • orb — ORB v3 conservative (Apache-2.0)
    • gpaw-pbe — GPAW PBE, PW(500 eV) (GPL-3.0)
    • gpaw-pbe-fast — GPAW PBE, PW(400 eV) (GPL-3.0)
🖥️ NVIDIA CUDA GPUs: 1
    • [CUDA 0] NVIDIA H100 80GB HBM3 — 79.1 GB, compute 9.0

                   __
   ____ ___  ____ _/ /__   _____  _____________
  / __ `__ \/ __ `/ __/ | / / _ \/ ___/ ___/ _ \
 / / / / / / /_/ / /_ | |/ /  __/ /  (__  )  __/
/_/ /_/ /_/\__,_/\__/ |___/\___/_/  /____/\___/

🔖 Version: 0.1.71   🧮 Functions: 218   📚 Tutorials: https://matverse.readthedocs.io/
✅ set_style complete.



## Loading a dataset

Three materials chosen to span the cases: a magnetic element, a magnetic alloy,
and something with no magnetic ion at all.

In [2]:
from pymatgen.core import Lattice, Structure


def fcc(symbol, a):
    return Structure(Lattice.cubic(a), [symbol] * 4,
                     [[0, 0, 0], [0, .5, .5], [.5, 0, .5], [.5, .5, 0]])


md = mv.data.from_structures([
    fcc("Ni", 3.524),
    Structure(Lattice.cubic(3.57), ["Ni", "Ni", "Ni", "Al"],
              [[0, 0, 0], [0, .5, .5], [.5, 0, .5], [.5, .5, 0]]),
    fcc("Al", 4.050),
])
mv.pp.standardize(md)
mv.pp.describe(md)

md.obs[["formula", "spacegroup", "nsites"]]

,formula,spacegroup,nsites
0,Ni,Fm-3m,4
1,AlNi3,Pm-3m,4
2,Al,Fm-3m,4


## Which elements can even carry a moment

In [3]:
mv.mag.describe(md)

md.obs[["formula", "magnetic_order", "n_magnetic_species",
        "total_magmom", "absolute_magmom"]].round(3)

,formula,magnetic_order,n_magnetic_species,total_magmom,absolute_magmom
0,Ni,unknown,1,NaN,NaN
1,AlNi3,unknown,1,NaN,NaN
2,Al,unknown,0,NaN,NaN


Read those two columns carefully, because they answer different questions.

`magnetic_order` is `unknown` for all three, and the moments are NaN. That is
correct: `mv.mag.describe` reports what the **structure carries**, and a CIF or
a hand-built cell carries no moments at all. It is not saying nickel is
non-magnetic; it is saying nobody has told this structure anything.

`n_magnetic_species` is the column doing the work here, and it comes from the
chemistry rather than from the file — 1 for the two nickel-bearing materials, 0
for aluminium. That is what decides whether enumerating orderings is even a
meaningful thing to do.

The set being tested against is `mv.mag.MAGNETIC_ELEMENTS`: the 3d, 4d and 5d
transition metals plus the lanthanides and actinides.

In [4]:
sorted(mv.mag.MAGNETIC_ELEMENTS)[:20]

['Ce',
 'Co',
 'Cr',
 'Cu',
 'Dy',
 'Er',
 'Eu',
 'Fe',
 'Gd',
 'Ho',
 'Mn',
 'Nd',
 'Ni',
 'Np',
 'Pr',
 'Pu',
 'Sm',
 'Tb',
 'Ti',
 'Tm']

## Enumerating orderings

`mv.mag.orderings` returns a **new object** whose rows are spin configurations,
with `obs['parent']` pointing back at the material — the same shape as
`mv.pp.defects` and `mv.surf.slabs`.

In [5]:
orderings = mv.mag.orderings(md, max_orderings=6)
orderings

AnnData object with n_obs × n_vars = 8 × 2
    obs: 'parent', 'ordering', 'ordering_index', 'total_magmom', 'is_magnetic'
    var: 'Z', 'atomic_mass', 'atomic_radius', 'electronegativity', 'group', 'period', 'melting_point', 'boiling_point', 'molar_volume', 'thermal_conductivity', 'electrical_resistivity', 'average_ionic_radius', 'max_oxidation_state', 'min_oxidation_state', 'is_metal', 'is_transition_metal', 'is_alkali', 'is_alkaline', 'is_metalloid', 'is_halogen', 'is_noble_gas', 'is_chalcogen', 'is_lanthanoid', 'is_actinoid', 'is_rare_earth_metal', 'block'
    uns: 'features', 'levels', 'provenance', 'X_is', 'magnetic_orderings'
    obsm: 'structures'
    layers: None (.X)

In [6]:
mv.pp.describe(orderings)
orderings.obs[["parent", "formula", "ordering", "ordering_index",
               "total_magmom", "is_magnetic"]].round(3)

,parent,formula,ordering,ordering_index,total_magmom,is_magnetic
0,0,Ni,fm,0,20.0,True
1,0,Ni,fim,1,10.0,True
2,0,Ni,afm,2,0.0,True
3,0,Ni,fim,3,-10.0,True
4,1,AlNi3,fm,0,15.0,True
5,1,AlNi3,fim,1,5.0,True
6,1,AlNi3,fim,2,-5.0,True
7,2,Al,nonmagnetic,0,0.0,False


Three things worth reading in that table.

The ferromagnetic ordering has the largest total moment, and the
antiferromagnetic one has **exactly zero** — that is what antiferromagnetic
means, and it is a check rather than a coincidence.

Aluminium comes through as a single `nonmagnetic` row rather than being dropped.
Every input material is still represented, so nothing needs re-joining by hand
afterwards.

And the moments are on the structures themselves, as a site property, so they
survive being written to disk and reach any calculator that knows what to do
with them.

In [7]:
mv.structures(orderings)[0].site_properties["magmom"]

[5.0, 5.0, 5.0, 5.0]

```{note}
pymatgen's antiferromagnetic enumeration calls out to **enumlib**, which is not
pip-installable and is absent from most environments. matverse falls back to a
simpler construction when it is missing — and records that it did.

Falling back is fine. Falling back silently is not: the fallback explores fewer
configurations, so a ground state found with it is a weaker claim than one found
with enumlib, and a reader has to be able to tell which they are looking at.
```

In [8]:
orderings.uns["magnetic_orderings"]["errors"]

["0: RuntimeError: EnumlibAdaptor requires the executables 'enum.x' or 'multienum.x' and 'makestr.x' or 'makeStr.py' to be in the path. Please download the library at https://github.com/msg-byu/enumlib and follow the instructions in the README to compile these two executables accordingly.; used the built-in fallback",
 "1: RuntimeError: EnumlibAdaptor requires the executables 'enum.x' or 'multienum.x' and 'makestr.x' or 'makeStr.py' to be in the path. Please download the library at https://github.com/msg-byu/enumlib and follow the instructions in the README to compile these two executables accordingly.; used the built-in fallback"]

## Picking the ground state

Compute every ordering, and let the lowest energy win. The winner's energy lands
back on the parent material.

In [9]:
mv.calc.energy(orderings, level="emt")
mv.mag.ground_state(orderings, md, level="emt")

md.obs[["formula", "magnetic_ordering_emt", "magnetic_spread_emt",
        "energy_per_atom_emt"]].round(6)

,formula,magnetic_ordering_emt,magnetic_spread_emt,energy_per_atom_emt
0,Ni,fm,0.0,-0.007592
1,AlNi3,fm,0.0,0.270962
2,Al,nonmagnetic,NaN,-0.001502


## The spread is zero, and that is the answer

`magnetic_spread` is the energy range across the orderings of one material, and
here it is **exactly zero** for both magnetic materials.

That is not a bug. EMT has no notion of spin at all, so every ordering of nickel
is the same set of atoms in the same positions and gets the same energy. The
calculator cannot distinguish them, and the object says so in a number rather
than by producing a confident arbitrary answer.

Which is the useful behaviour: `magnetic_ordering_emt` says `fm` for nickel, and
`magnetic_spread_emt` says that claim is worth nothing.

In [10]:
md.uns["magnetic"]["emt"]

{'n_with_alternatives': 2,
 'max_spread': 0.0,
 'collinear': True,
 'note': 'magnetic_spread is the gap between the best and worst ordering; a large one means the hull depends on this choice'}

```{warning}
**This is where the tutorial stops, because the calculator that ships with
matverse cannot go further.** Resolving a magnetic ground state needs a
spin-polarised method — DFT with initialised moments, or a machine-learned
potential trained on magnetic configurations.

The enumeration above is real and reusable. The energies are not.
```

With such a calculator registered, the rest is unchanged:

```python
from mace.calculators import mace_mp

mv.calc.register_calculator("mace-mpa", lambda: mace_mp(model="medium-mpa-0"),
                            kind="mlip", method="MACE-MPA-0",
                            reference="PBE+U", license="MIT")

orderings = mv.mag.orderings(md, max_orderings=8)
mv.calc.relax(orderings, level="mace-mpa")
mv.mag.ground_state(orderings, md, level="mace-mpa")
```

or by writing the orderings out for DFT:

```python
mv.dft.write_inputs(orderings, code='vasp', preset='relax',
                    directory='orderings/')
```

## Why this belongs before the hull

The energy `mv.mag.ground_state` writes back is under the **ordinary** column
name the calculator would have produced — `energy_per_atom_emt`, not
`energy_per_atom_emt_magnetic`.

In [11]:
[c for c in md.obs.columns if c.startswith("energy")]

['energy_per_atom_emt']

That is deliberate. `mv.thermo.hull` needs no special case for magnetism: it
sees a normal energy column that happens to be the magnetic ground state, and a
pipeline written without magnetism in mind keeps working when magnetism is
added in front of it.

The alternative — a specially-named column — would mean every downstream
function needed to know about magnetic ordering, and the ones that did not know
would silently use the wrong energy.

```{seealso}
[Screening, end to end](screening.ipynb) is the pipeline this step goes in front
of. [Defects and diffusion](defects_and_diffusion.ipynb) has the same shape:
enumerate configurations, compute them all, let the object record which won.
```

## The other thing a d shell does

Magnetic ordering is one consequence of a partly filled d shell. Distortion is
the other, and it is structural rather than magnetic: a degenerate electronic
ground state in an octahedral site lowers its energy by distorting the
octahedron.

That is not a detail. It is why LaMnO₃ is orthorhombic rather than cubic, and
why manganese spinel cathodes fade on cycling.

In [12]:
from pymatgen.core import Lattice, Structure

def perovskite(a_site, b_site, a):
    return Structure(Lattice.cubic(a), [a_site, b_site, "O", "O", "O"],
                     [[0, 0, 0], [.5, .5, .5], [.5, .5, 0],
                      [.5, 0, .5], [0, .5, .5]])

perovskites = mv.data.from_structures([perovskite("La", "Mn", 3.9),
                                       perovskite("Sr", "Ti", 3.905),
                                       perovskite("La", "Ni", 3.85)])
mv.pp.describe(perovskites)
mv.mag.jahn_teller(perovskites)

perovskites.obs[["formula", "jahn_teller_active", "jahn_teller_strength",
                 "jahn_teller_species"]]

,formula,jahn_teller_active,jahn_teller_strength,jahn_teller_species
0,LaMnO3,True,strong,Mn3+
1,SrTiO3,False,none,
2,LaNiO3,True,strong,Ni3+


Mn³⁺ and Ni³⁺ come out **strong** and Ti⁴⁺ inactive, which is the textbook
answer: both of the first two put an electron in a doubly degenerate e_g level,
and Ti⁴⁺ is d⁰ with no degeneracy to lift.

The column that matters is `jahn_teller_species` — knowing a material distorts
is not useful without knowing which site is doing it. The ligand bond lengths,
which are the distortion itself rather than a label for it, stay in `uns`.

```{warning}
`strong` means an e_g degeneracy and `weak` a t₂_g one, and the difference is
large: a weak Jahn-Teller distortion usually does not survive room temperature.
Reading `weak` as `distorted` is the common mistake.

The answer also depends on the spin state, which is guessed only if you ask
with `guess_spin=True`. A structure that already carries oxidation states from
`mv.transform.oxidation_states` is used as given rather than re-assigned.
```

## How far above room temperature does it stay a magnet?

Which ordering is lowest says whether a material is a ferromagnet or an
antiferromagnet. It says nothing about how hot it can get before it stops being
one — and that is usually the question a screen is really asking.

Mapping the ordering energies onto a Heisenberg Hamiltonian gives the exchange
couplings, and a mean-field estimate of the ordering temperature follows:

In [13]:
from pymatgen.core import Lattice, Structure

def iron(moments):
    st = Structure(Lattice.cubic(2.87), ["Fe", "Fe"],
                   [[0, 0, 0], [.5, .5, .5]])
    st.add_site_property("magmom", list(moments))
    return st

# stand-in for two spin-polarised total energies, 80 meV apart
spins = mv.data.from_structures([iron([1.0, 1.0]), iron([1.0, -1.0])])
spins.obs_names = ["ferromagnetic", "antiferromagnetic"]
spins.obs["energy_pbe"] = [-0.08, 0.08]

mv.mag.exchange(spins, level="pbe", cutoff=3.0)

spins.obs[["energy_pbe", "exchange_pbe", "ordering_temperature_pbe"]].round(3)

,energy_pbe,exchange_pbe,ordering_temperature_pbe
ferromagnetic,-0.08,80.0,618.908
antiferromagnetic,0.08,80.0,618.908


The ferromagnetic arrangement is lower, so the coupling is positive and the
material orders ferromagnetically — and it does so up to roughly 620 K on this
estimate.

```{warning}
**Read that temperature as an upper bound.** Mean-field theory ignores exactly
the fluctuations that destroy magnetic order, so it overestimates Curie and Néel
temperatures systematically — often by a third to a half. It is good for ranking
candidates and for ruling things out ("nowhere near room temperature"); it is
not a prediction of a measurement.

The couplings come back in meV under pymatgen's convention, which counts per
site rather than per bond. Ratios between materials are convention-free; the
absolute number is not.
```

### When the fit has nothing to fit

This only means anything for energies from a **spin-polarised** calculation. A
potential that does not distinguish spin returns the same energy for every
ordering, and the Heisenberg fit is then degenerate. matverse says so instead of
reporting a small coupling:

In [14]:
import warnings

flat = spins.copy()
flat.obs["energy_pbe"] = [-1.0, -1.0]      # what a spin-blind potential gives

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    mv.mag.exchange(flat, level="pbe", cutoff=3.0)

print(str(caught[-1].message)[:150])
flat.uns["exchange"]["pbe"]["error"]

mv.mag.exchange could not fit couplings: every ordering has the same energy to within 0.00e+00 eV, so the Heisenberg fit is degenerate — this is what 


'every ordering has the same energy to within 0.00e+00 eV, so the Heisenberg fit is degenerate — this is what a calculator that does not distinguish spin produces'

A NaN and a stated reason, rather than a number near zero that would read as
"weakly coupled" when it actually means "not calculated with spin".

## What the moments cost in symmetry

Putting moments on a lattice breaks some of its symmetry and leaves the rest.
Which is which is what a magnetic space group records — and pymatgen ships all
1651 of them but **no analyser that reads one off a structure**. There is no
`MagneticSpaceGroupAnalyzer` beside `SpacegroupAnalyzer`.

`mv.mag.symmetry` computes the underlying quantity instead of naming the group:
every operation of the non-magnetic parent is applied to the moments as the
axial vectors they are, once plainly and once with time reversal, and an
operation survives if either version maps the arrangement onto itself.

In [15]:
from pymatgen.core import Lattice, Structure

def iron_with(moments):
    st = Structure(Lattice.cubic(2.87), ["Fe", "Fe"],
                   [[0, 0, 0], [.5, .5, .5]])
    st.add_site_property("magmom", list(moments))
    return st

arrangements = mv.data.from_structures([
    iron_with([0.0, 0.0]), iron_with([2.2, 2.2]), iron_with([2.2, -2.2])])
arrangements.obs_names = ["no moments", "ferromagnetic", "antiferromagnetic"]

mv.mag.symmetry(arrangements)
arrangements.obs[["parent_symmetry_order", "magnetic_symmetry_order",
                  "magnetic_symmetry_fraction"]]

,parent_symmetry_order,magnetic_symmetry_order,magnetic_symmetry_fraction
no moments,96.0,96.0,1.000000
ferromagnetic,96.0,32.0,0.333333
antiferromagnetic,96.0,32.0,0.333333


bcc iron has **96** operations. With no moments all 96 survive. With any
collinear ordering along *z*, only the **32** that leave that axis alone do —
a third of the crystal's symmetry, gone, from adding a property that changes no
atomic position.

That fraction is worth having in a screen: one means the moments cost nothing,
a small number means the ordering has broken most of the symmetry, and that is
where to expect magnetic anisotropy and where two orderings at the same energy
are not the same state.

```{note}
**Ferromagnet against antiferromagnet is deliberately not reported here.** The
clean statement of that is whether the magnetic space group contains pure time
reversal, and counting primed operations is not the same thing — a collinear
ferromagnet has primed operations too, picked up from rotations that reverse
its axis. Getting it right needs the group type, which needs the analyser that
does not exist. `mv.mag.describe` gives the net moment, which answers the
practical question without dressing itself up as a symmetry classification.
```